# Przeszukiwanie przestrzeni stanów: BFS i DFS – interaktywnie
**Wprowadzenie do AI · laboratorium 3**

Plan:
0. **Powtórka z poprzednich zajęć** – od przestrzeni stanów i algorytmu Dijkstry do przeszukiwania wszerz (BFS),
1. **Jak wędrować po grafie?** – wizualizacja i animacja **BFS (wszerz)** oraz **DFS (w głąb)** na grafach z Lab 2 (winda, odkurzacz, pojemniki) + **ćwiczenie „na kartce”**: BFS, DFS i najkrótsza ścieżka na losowym grafie (klikasz węzły, program sprawdza),
2. **Dwa sposoby implementacji (Iteracja vs Rekurencja)** – jawna **kolejka FIFO / stos LIFO** kontra **stos wywołań funkcji (*Call Stack*)**,
3. **Od klasycznego AI do współczesnych modeli językowych (LLM)** – jak przeszukiwanie drzew (BFS i DFS) służy do śledzenia **głowic uwagi (*Attention Heads*)** w Transformerach.

> **Jak uruchomić:** uruchom komórki po kolei (`Shift+Enter`). Pierwsza komórka (ukryta) sama pobierze potrzebny kod z GitHuba – nic nie trzeba instalować ani wgrywać.

In [ ]:
#@title ⚙️ Przygotowanie – uruchom tę komórkę (kod ukryty)
import os, sys, subprocess
REPO = "https://github.com/BartekDaniluk/kognitywistyka_intro_to_AI"
if "google.colab" in sys.modules:
    root = "/content/kognitywistyka_intro_to_AI"
    if not os.path.exists(root):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, root], check=True)
    else:
        subprocess.run(["git", "-C", root, "pull", "-q"], check=False)
    for sub in ("lab02_stany", "lab03_przeszukiwanie"):
        p = os.path.join(root, sub)
        if p not in sys.path:
            sys.path.insert(0, p)
else:
    for sub in (".", "../lab02_stany", "lab02_stany", "lab03_przeszukiwanie"):
        p = os.path.abspath(sub)
        if os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)
%matplotlib inline
import lab_states as ls
import lab_search as lsearch
ls.set_language("pl")
print("Gotowe ✔")

## 0. Powtórka: co już wiemy i jak działał solver z poprzednich zajęć?

Na poprzednim laboratorium nauczyliśmy się **formułować problemy** za pomocą pięciu składników:

| Składnik | Co opisuje? | Przykład (Winda) | Odpowiednik w grafie |
|---|---|---|---|
| **1. Stany ($S$)** | Opis sytuacji (krotka zmiennych, z pominięciem nieistotnych szczegółów – *abstrakcja*) | `(P, D)` – piętro i stan drzwi | **Wierzchołki (węzły)** grafu |
| **2. Stan początkowy ($s_0$)** | Sytuacja, od której zaczynamy | `(0, o)` | Węzeł startowy (zielona ramka) |
| **3. Test celu** | Warunek mówiący, czy osiągnęliśmy sukces | `(2, o)` | Węzły docelowe (żółte tło) |
| **4. Akcje i model przejścia** | Reguły `(stan \| warunek) → nowy stan` | Reguły 1–4 (góra, dół, otwórz, zamknij) | **Skierowane krawędzie (strzałki)** |
| **5. Koszt akcji ($c$)** | Ile „płacimy” za wykonanie kroku | Każda akcja kosztuje `1` | Wagi krawędzi |

### Tajemnica funkcji `solve()` z Lab 2: Dijkstra o równych wagach to po prostu BFS!

W pierwszym notebooku program sam podświetlał na niebiesko **rozwiązanie optymalne**. Pod maską używał do tego **algorytmu Dijkstry** (w sztucznej inteligencji nazywanego **UCS – *Uniform Cost Search***, czyli *przeszukiwaniem z jednolitym kosztem*).

Jak działa Dijkstra? Zawsze wybiera do sprawdzenia ten stan, do którego prowadzi **najtańsza dotychczasowa ścieżka od startu** (oznaczana jako $g(s)$).

Zastanówmy się jednak: **co się dzieje, gdy każda akcja ma dokładnie taki sam koszt równy $1$?**
* Koszt dojścia $g(s)$ jest wtedy po prostu równy **liczbie wykonanych kroków**, czyli **głębokości $d(s)$** w grafie!
* Algorytm sprawdza więc najpierw stan na głębokości $d=0$ (start), potem wszystkie stany oddalone o $d=1$ krok, potem wszystkie oddalone o $d=2$ kroki itd.
* Takie przeszukiwanie **warstwa po warstwie (falami)** to właśnie **BFS (*Breadth-First Search* – przeszukiwanie wszerz)**!

Sprawdźmy to na żywo na znanym z poprzednich zajęć grafie windy:

In [ ]:
# Porównanie: BFS (wszerz) vs Dijkstra z Lab 2 (przy koszcie każdej akcji = 1)
lsearch.dijkstra_vs_bfs_demo()

## 1. Przeszukiwanie grafu: BFS (wszerz) kontra DFS (w głąb)

Kiedy człowiek patrzy na mały graf narysowany na kartce, od razu widzi całość „z lotu ptaka” i wzrokiem znajduje drogę do celu.
Komputer (oraz człowiek rozwiązujący duży problem, którego nie da się objąć wzrokiem) działa inaczej: **stoi w jednym konkretnym stanie** i za pomocą reguł odkrywa jego bezpośrednich sąsiadów.

Żeby systematycznie przeszukać przestrzeń stanów i nie kręcić się w kółko (np. bez końca otwierać i zamykać drzwi na parterze: `(0, o) ↔ (0, z)`), algorytm potrzebuje dwóch list:
1. **Odwiedzone (`Visited` / lista zamknięta)** – zbiór stanów, które już sprawdziliśmy (na grafie: **niebieskie węzły** z numerem kroku). Jeśli reguła prowadzi do stanu, który już tu jest – ignorujemy go!
2. **Oczekujące (`Frontier` / lista otwarta)** – stany, które odkryliśmy, ale jeszcze z nich „nie wychodziliśmy” (na grafie: **fioletowe węzły** z napisem `czeka`).

---

### Cała różnica między BFS a DFS to jedna zasada: *skąd bierzemy kolejny stan?*

| Cecha | 🌊 BFS (*Breadth-First Search* – wszerz) | 🕳️ DFS (*Depth-First Search* – w głąb) |
|---|---|---|
| **Struktura danych** | **Kolejka FIFO** (*First-In, First-Out*) | **Stos LIFO** (*Last-In, First-Out*) |
| **Metafora z życia** | Kolejka w sklepie: kto pierwszy stanął w kolejce, ten pierwszy jest obsłużony | Stos talerzy: kładziemy nowy talerz na samą górę i z samej góry go zdejmujemy |
| **Jak wędruje po grafie?** | **Rozlewa się falami (warstwa po warstwie)**: najpierw sprawdza wszystkie stany w odległości $d=1$, potem $d=2$, potem $d=3$… | **Nurkuje w głąb jedną ścieżką** tak daleko, jak się da; gdy trafi na ślepy zaułek, **cofa się (*backtracking*)** do ostatniego rozwidlenia |
| **Czy znajdzie najkrótszą ścieżkę?** | **TAK** (zawsze znajduje rozwiązanie o najmniejszej liczbie kroków) | **NIE** (może znaleźć cel bardzo szybko albo okrężną, długą drogą) |
| **Zużycie pamięci** | Duże (musi pamiętać całą szeroką „falę” stanów) | Małe (pamięta głównie aktualną ścieżkę i jej rozgałęzienia) |

### 🎬 Interaktywny symulator i animacja
* Wybierz problem z Lab 2 (**Winda**, **Świat odkurzacza** lub **Pojemniki z wodą**) oraz algorytm.
* Kliknij **▶ (Play)** albo przesuwaj suwak **Krok**, obserwując jednocześnie:
  * **po lewej:** jak algorytm odwiedza kolejne węzły grafu i buduje drzewo przeszukiwania,
  * **po prawej:** jak elementy trafiają do **kolejki FIFO**, na **stos LIFO** lub na **stos wywołań rekurencyjnych**,
  * **w zakładce ⚖️ Porównanie:** zestawienie BFS, DFS i Dijkstry obok siebie,
  * **w zakładce 📝 Tabela kroków:** pełny zapis przebiegu dokładnie w takiej formie, w jakiej tradycyjnie rozpisuje się BFS i DFS „na kartce”.

In [ ]:
lsearch.search_app()

### ✏️ Zadania do sprawdzenia w symulatorze
1. **Winda (3 piętra):** porównaj przebieg **BFS** i **DFS** (zobacz też zakładkę *⚖️ Porównanie*). Czy na tym grafie oba algorytmy odwiedzają stany w tej samej kolejności, czy w innej? Dlaczego?
2. **Świat odkurzacza (2 pola oraz 3 pola):**
   * Uruchom **BFS**, a następnie **DFS (stos LIFO)** dla *Świata odkurzacza (2 pola)*.
   * Ile kroków ma ścieżka znaleziona przez BFS, a ile przez DFS? Który algorytm znalazł rozwiązanie optymalne?
   * Włącz *Tryb szczegółowy* i zobacz, co robi algorytm, gdy z jakiegoś stanu akcja prowadzi do stanu już wcześniej odwiedzonego (np. ruch w lewo albo powrót na poprzednie pole). Co by się stało, gdybyśmy **nie** pamiętali zbioru `odwiedzone`?
3. **Pojemniki z wodą (4 l i 3 l):**
   * Przełącz problem na *Pojemniki z wodą (4 l i 3 l, koszt=1)* i porównaj na zakładce *⚖️ Porównanie* drzewo przeszukiwania **BFS** i **DFS**.
   * Zwróć uwagę na to, jak BFS schodzi równomiernie po dwóch gałęziach grafu (zaczynając od reguły 1 albo od reguły 2), podczas gdy DFS od razu brnie jedną gałęzią w głąb.
   * Teraz zmień problem na *Pojemniki z wodą (koszt=litry wody)* i porównaj **BFS** z **Dijkstrą / UCS**. Dlaczego teraz Dijkstra znajduje inne rozwiązanie niż BFS?

### 🎲 Ćwiczenie: BFS i DFS „na kartce” – na losowym grafie

Symulator robi wszystko za Ciebie. Teraz **Ty jesteś algorytmem**: program losuje graf (do 10 węzłów, numery 0–9),
węzeł **START** (zielona ramka) i węzeł **CEL** (żółty). Twoje zadania:

1. **BFS** – zapisz kolejność, w jakiej BFS odwiedza węzły,
2. **DFS** – to samo dla DFS,
3. **Najkrótsza ścieżka** – zapisz najkrótszą drogę ze START do CEL.

**Jak to działa:** kliknij zadanie po prawej (kolorowa kropka = zadanie aktywne), a potem **klikaj węzły na grafie** w odpowiedniej
kolejności – ich numery pojawiają się na liście i przy węzłach. `⌫` cofa ostatni węzeł, `✕` czyści listę.
Gdy skończysz, kliknij **✔ Sprawdź**: dobre odpowiedzi zrobią się zielone 😀, błędne czerwone 😢 (z podpowiedzią, gdzie jest pierwszy błąd).

**Zasady** (dzięki nim jest dokładnie jedna poprawna kolejność):
* sąsiadów rozpatrujemy zawsze w kolejności **rosnących numerów**,
* odwiedzamy **cały graf** – nie zatrzymujemy się na celu,
* DFS w wersji rekurencyjnej (daje tę samą kolejność co stos LIFO z symulatora).

> 💡 **Wskazówka:** rozwiązuj z kartką obok – zapisuj zawartość **kolejki** (BFS) albo **stosu** (DFS) po każdym kroku, tak jak w symulatorze.
> **Numer grafu** pozwala całej grupie pracować na tym samym grafie (wpisz ten sam numer), a **🎲 Nowy graf** losuje kolejny.

In [ ]:
import lab_traversal_quiz as quiz
quiz.traversal_exercise(n_nodes=8)   # n_nodes: 5–10; seed=1234 → ten sam graf dla wszystkich

#### ✏️ Pytania do ćwiczenia
1. Rozwiąż bezbłędnie **trzy różne grafy** (przycisk *🎲 Nowy graf*). Zacznij od 6 węzłów, potem spróbuj 10.
2. Porównaj **kolejność odwiedzania BFS** z **najkrótszą ścieżką**. Dlaczego ścieżka to *nie* jest po prostu początek listy BFS?
   Jak z przebiegu BFS odczytać ścieżkę? *(wskazówka: zapamiętuj, który węzeł „odkrył” który – to tzw. rodzic)*
3. Znajdź graf, w którym **DFS** dociera do CEL **wcześniej** (na wcześniejszej pozycji listy) niż BFS. Czy to znaczy, że DFS znalazł krótszą drogę?
4. Ustaw *Krawędzi: mało*. Kiedy BFS i DFS dają **tę samą** kolejność odwiedzania? Narysuj na kartce graf, dla którego tak będzie.

## 2. Dwa sposoby implementacji BFS i DFS: Iteracja vs Rekurencja

W programowaniu (i w modelowaniu procesów poznawczych!) zarówno **DFS**, jak i **BFS** można zaimplementować na **dwa fundamentalnie różne sposoby**:

### Sposób 1: Iteracyjny (pętla `while` + własna struktura danych)
Sami tworzymy zmienną przechowującą stany oczekujące na sprawdzenie i w pętli `while` wyjmujemy z niej po jednym elemencie:
* Dla **BFS**: używamy **kolejki FIFO** (`kolejka = deque([start])`), wyjmujemy z przodu (`kolejka.popleft()`), dopisujemy na koniec (`kolejka.append(s2)`).
* Dla **DFS**: używamy **stosu LIFO** (`stos = [start]`), zdejmujemy z wierzchu (`stos.pop()`) i odkładamy na wierzch (`stos.append(s2)`).

### Sposób 2: Rekurencyjny (funkcja wywołuje samą siebie + Stos wywołań *Call Stack*)
Zamiast pętli `while` piszemy funkcję, która **wywołuje samą siebie** dla kolejnych stanów:
* **DFS rekurencyjny (`dfs(s)`)**: w kodzie nie widać żadnej zmiennej `stos`! Gdzie więc podział się stos?
  * Ukrywa się on w pamięci komputera jako **stos wywołań funkcji (*Call Stack*)**.
  * Za każdym razem, gdy funkcja `dfs(s)` wywołuje `dfs(s2)` dla sąsiada, Python „zamraża” bieżące wywołanie `dfs(s)` i kładzie na wierzchu stosu wywołań nową **ramkę (*stack frame*)** dla `dfs(s2)`.
  * Gdy `dfs(s2)` skończy sprawdzać swoich sąsiadów (trafi na ślepy zaułek), wykonuje `return` – jego ramka jest zdejmowana ze stosu wywołań, a program automatycznie wraca (*backtrack*) dokładnie do miejsca, w którym czekało `dfs(s)`!
* **BFS rekurencyjny (`bfs_warstwa(warstwa, d)`)**: ponieważ zwykłe wywołanie rekurencyjne naturalnie działa jak stos (w głąb), w rekurencyjnym BFS przekazujemy jako argument **całą listę stanów z bieżącej warstwy $d$**. Funkcja przegląda wszystkie stany z warstwy $d$, zbiera ich sąsiadów do listy `nastepna_warstwa` i na samym końcu wywołuje samą siebie dla warstwy $d+1$: `bfs_warstwa(nastepna_warstwa, d + 1)`.

Sprawdź poniżej krok po kroku, jak **jawny stos / kolejka** w wersji iteracyjnej odpowiada **ramkom na stosie wywołań (*Call Stack*)** w wersji rekurencyjnej:

In [ ]:
lsearch.recursion_vs_iteration_app()

### 💡 Kiedy używać iteracji, a kiedy rekurencji?

| Kryterium | Wersja iteracyjna (`while` + Stos / Kolejka) | Wersja rekurencyjna (wywołania funkcji / *Call Stack*) |
|---|---|---|
| **Czytelność kodu** | Naturalna i najprostsza dla **BFS** | Bardzo zwięzła i naturalna dla **DFS** oraz przeszukiwania drzew |
| **Pamięć i bezpieczeństwo** | Bezpieczna nawet dla bardzo głębokich grafów (ogranicza nas tylko pamięć RAM) | W Pythonie stos wywołań ma domyślny limit ok. **1000 wywołań w głąb** (`RecursionError`) |
| **Kontrola nad przeszukiwaniem** | Łatwo zatrzymać, wznowić lub zmienić w Dijkstrę/A* (wystarczy zamienić kolejkę na kopiec!) | Automatyczne cofanie się (*backtracking*) po zakończeniu funkcji (`return`) |

## 3. Od klasycznego przeszukiwania do współczesnych modeli AI: Przeszukiwanie głowic uwagi (*Attention Heads*)

Czy przeszukiwanie drzew i grafów za pomocą BFS i DFS to tylko historia z czasów klasycznej sztucznej inteligencji i prostych łamigłówek?
**Wręcz przeciwnie!** Dokładnie tych samych algorytmów używamy dziś do zaglądania „do wnętrza umysłu” współczesnych sieci neuronowych – takich jak **Transformery** (modele GPT, Claude, Gemini czy LLaMA).

### Jak powstaje drzewo uwagi w Transformerze?
1. Gdy model językowy przetwarza zdanie (np. *„Jan dał książkę Marii, bo **ona**...”*), słowa (tokeny) przechodzą przez kolejne **warstwy sieci** ($L_0 \to L_1 \to L_2 \to L_3 \dots$).
2. W każdej warstwie działa wiele **głowic uwagi (*Attention Heads*)**. Każda głowica uczy się zwracać uwagę na inne relacje między słowami:
   * jedna głowica łączy czasownik z podmiotem (*kto wykonał czynność?*),
   * inna rozwiązuje zaimki (*do kogo odnosi się słowo „ona”?*),
   * jeszcze inna (tzw. **głowica indukcyjna – *Induction Head***) szuka powtarzających się wzorców we wcześniejszym tekście.
3. Ponieważ token w warstwie $L_3$ pobiera informacje z tokenów w warstwie $L_2$, które z kolei pobrały informacje z warstwy $L_1$ (a te z warstwy wejściowej $L_0$), połączenia uwagi tworzą **wielowarstwowe drzewo / graf przepływu informacji**!

### Po co przeszukujemy głowice uwagi za pomocą BFS i DFS?
W dziedzinie **interpretowalności mechanistycznej (*Mechanistic Interpretability*)** chcemy zrozumieć, **dlaczego** model wygenerował dane słowo:
* 🌊 **BFS na drzewie uwagi (wszerz, warstwa po warstwie – *Attention Rollout / Attribution*)**:
  Zaczynamy od badanego tokenu wyjściowego (np. `[L3] „ona”`) i za pomocą **kolejki FIFO** schodzimy w dół **warstwa po warstwie** ($L_3 \to L_2 \to L_1 \to L_0$). Pozwala to zobaczyć pełny „rozpływ” uwagi na każdym poziomie głębokości.
* 🕳️ **DFS na drzewie uwagi (w głąb, ścieżka po ścieżce – *Circuit Tracing*)**:
  Za pomocą **rekurencji / stosu** (oraz odrzucania słabych połączeń poniżej progu $\alpha$, tzw. *pruningu*) śledzimy od początku do końca pojedyncze **obwody neuronalne (*circuits*)** – czyli konkretne łańcuchy przyczynowe łączące predykcję modelu z konkretnym słowem na wejściu.

Sprawdź to na trzech przykładach poniżej:

In [ ]:
lsearch.attention_traversal_app()

### ✏️ Pytania i eksperymenty (Głowice uwagi)
1. **Rozwiązywanie zaimków (*„Jan dał książkę Marii, bo ona...”*):**
   * Ustaw próg wagi uwagi na `$\alpha \ge 0.20$` i porównaj kolejność odwiedzania węzłów dla **BFS** oraz **DFS** (przesuwaj suwak *Krok* od 0 do końca).
   * Która głowica uwagi odpowiada za bezpośrednie połączenie słowa `„ona”` z `„Marii”`, a która za połączenie przez czasownik `„dał”`?
   * Zwiększ próg uwagi do `$\alpha \ge 0.50$` (odcięcie słabszych krawędzi – *pruning*). Który obwód (*circuit*) pozostał w drzewie?
2. **Obwód indukcyjny (*„Kognitywistyka bada umysł. AI bada → [?]”*):**
   * Przełącz na przykład 2. Zobacz za pomocą **DFS**, jak dwie głowice współpracują na różnych głębokościach drzewa: w warstwie L2 *H1 (Induction Head)* przenosi uwagę z drugiego słowa `„bada”` na `„umysł”`, ponieważ w warstwie L1 *H0 (Previous Token Head)* zapisała w reprezentacji słowa `„umysł”` informację, że stało ono zaraz po pierwszym słowie `„bada”`!
3. **Wnioskowanie wielokrokowe (*Multi-hop*):**
   * Przełącz na przykład 3 (*„Klucz jest w skrzyni, skrzynia w piwnicy...”*). Dlaczego model nie jest w stanie rozwiązać tego zadania w jednej warstwie i potrzebuje przejścia w głąb przez dwie warstwy głowic uwagi?

## 4. Dla ciekawskich: uruchamianie BFS i DFS bezpośrednio z kodu

Wszystkie algorytmy możesz też wywołać bezpośrednio w kodzie Pythona dla dowolnego problemu z Lab 2 i wypisać ich kroki:

In [ ]:
problem = ls.WaterJugs(cap_a=4, cap_b=3, initial=(0, 0), goal=(2, 0))

for nazwa, algo in [("BFS iteracyjny", "bfs_iter"), ("DFS iteracyjny", "dfs_iter"), ("DFS rekurencyjny", "dfs_rec")]:
    kroki = lsearch.run_trace(problem, algo=algo)
    ostatni = kroki[-1]
    odwiedzone = " -> ".join(problem.state_label(s) for s in ostatni.visited)
    sciezka = problem.solution_text(ostatni.solution or [])
    print(f"{nazwa:18s} | odwiedzono {len(ostatni.visited):2d} stanów | ścieżka ({len(ostatni.solution)} kr.): {sciezka}")
    print(f"   kolejność: {odwiedzone}\n")